# A1: Outputs

How do decision models behave on the same requests? Five views of the 17 decision models' stored probabilities: who the models are and what they cost to run, the shape of their outputs, how alike two models are, how close each model is to the others' consensus, and how good and calibrated each model is on its own. Binary requests give one probability, multiclass requests one probability per class, regression requests nine probabilities over ordered evidence levels.

All tables come from **one compact export**, `output_JEVPFN/exp01_outputs/results/outputs/comparison.zip`, built on VSC by the Mindwell CPU job `scripts/slurm/rq1_outputs.slurm` (`python -m src.evaluation.comparison_report`) from the complete feature stores on PROJECT. This notebook reads only that ZIP and recomputes nothing.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.config import save_resolved_config
from src.utils.notebook_display import display_local, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.evaluation import comparison_report as report, model_properties, study_export, zero_shot
from src.visualize import figures, style, outputs as plots
from src.visualize.zero_shot import direct_score_figures
style.apply()
NOTEBOOK = "exp01_outputs"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
cfg = report.report_config()
save_resolved_config(cfg, NOTEBOOK)
report_section("1. The export read by this notebook")
result, receipt = report.read_report(cfg, allow_previous_roster=True)
available = receipt["status"].startswith("verified")
print(report.describe_receipt(receipt))
if receipt.get("unassessed_additions"):
    print("Models awaiting a new export:", ", ".join(receipt["unassessed_additions"]))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
if available:
    inventory = result.table("inventory")
    blocks = (inventory.assign(available=inventory.status.eq("available")).groupby("provider", sort=False)["available"]
              .agg(["sum", "size"]).rename(columns={"sum": "Complete blocks", "size": "Expected blocks"}))
    blocks.index = [style.DECISION_SHORT.get(p, p) for p in blocks.index]
    display_local(blocks.rename_axis("Decision model").reset_index())
    print("A block is one dataset and one request mode (per-column or joint). Only models with every block enter the comparison.")
    record(inventory, "Exact immutable feature artifact identities")

comparison.zip: created 2026-11-18T13:24:39.664228+00:00 by verified local export of 11 complete VSC models, 2026-11-18, 1,912,238 bytes, SHA-256 89c4986f83ba69546646a33b09ee4aa4ed8d94f9bd95f2d17050822d288cb8dd. Schema jevpfn.rq1a.report.v1 predates jevpfn.rq1a.report.v2; waiting for a new export. No measurements are shown.


## 1. Model profile

Who are the models? Size, readout type and architecture family come from the sourced properties file (`config/02_feature_creation/model_properties.yaml`); a value no local source states stays missing. Compute time per 1,000 decision requests is the provider's successful decision-feature GPU-hours in the Slurm compute ledger divided by its number of distinct requests in the 0A token audit, ×1,000. It is compute time only, with no token or API prices; a model without a recorded successful GPU job or an audited request count (for example the hosted Jev API) stays missing.

In [2]:
report_section("2. Model profile")
profile = result.table("model_profile") if available else pd.DataFrame()
if profile.empty:
    profile = model_properties.profile_table()
    print("Compute time is waiting for the export; properties are read from the configuration.")
shown = profile.assign(model=profile.provider.map(lambda p: style.DECISION_SHORT.get(p, p)))
display_table("Model profile: size, readout, architecture and GPU-hours per 1,000 requests (missing = not recorded)",
              shown.drop(columns=["provider"]).set_index("model").round(4).reset_index(), preview_rows=17)
record(profile, "Model profile")

Compute time is waiting for the export; properties are read from the configuration.


model,display_name,parameters_billion,readout_type,architecture_family,gpu_hours_per_1000_requests,compute_status
Jev 1.14,Jev 1.14,NaN,api,undisclosed,NaN,waiting for export
Liquid d1-3B,d1-3B,3.120,lm_label_tokens,decoder_lm_head,NaN,waiting for export
Liquid d1-2B,d1-2B,NaN,None,None,NaN,waiting for export
Perplexity 27B,pplx-decider-v1.1-27b,26.085,linear_readout,bidirectional_encoder,NaN,waiting for export
Clef-flash,clef-flash,9.410,pointer_head,decoder_pointer_head,NaN,waiting for export
Intern 4B,Intern-Decision-4B,NaN,lm_label_tokens,decoder_lm_head,NaN,waiting for export
Strands 2B,strands-decider-2B-qwen3.5-v1-2610,NaN,pointer_head,decoder_pointer_head,NaN,waiting for export
H2O 4B,h2o-lightning-4b,4.660,lm_label_tokens,bidirectional_encoder,NaN,waiting for export
SII 27B,SII-Decision-27B-Full,NaN,lm_label_tokens,decoder_lm_head,NaN,waiting for export
Kev 27B,kev-27b,25.600,pointer_head,decoder_pointer_head,NaN,waiting for export


## 2. Output shape

How concentrated are the answers? Per model, dataset and request mode: the mean top probability, the share of answers whose top probability exceeds 0.95 and the normalized entropy (0 certain, 1 uniform). For regression, the spread over the nine levels (standard deviation of the level under the model's probabilities) and the distribution of the expected level. Each distinct input counts once; per-column text fields are averaged with equal weight within a dataset.

In [3]:
report_section("3. Output shape")
shape = result.table("output_shape") if available else pd.DataFrame()
if shape.empty:
    print("Waiting for export: no output-shape table is available.")
else:
    for mode in ("joint", "per_column"):
        for fig, name, caption in [*plots.output_shape_figure(shape, mode), *plots.regression_levels_figure(shape, mode)]:
            save(fig, name, caption=caption)
            display_local(fig)
            plt.close(fig)
    table = report.dataset_mode_table(shape, report.OUTPUT_SHAPE_METRICS)
    display_table("Output shape per model, dataset and request mode", table.round(3), preview_rows=8)
    record(table, "Output shape per model, dataset and request mode")
    record(result.table("level_histogram"), "Distribution of the expected regression level (bins of 0.5 levels)")

Waiting for export: no output-shape table is available.


## 3. Pairwise agreement

Are two models interchangeable? On the same request identities: the share of inputs on which both give the same top answer (an exact tie for the top answer abstains), and the distance between their probability vectors: total variation for binary and multiclass, earth mover's distance over the nine ordered levels for regression. One matrix per task, equal dataset weight.

In [4]:
report_section("4. Pairwise agreement")
pairs = result.table("pairs") if available else pd.DataFrame()
if pairs.empty or "top_agreement" not in pairs:
    print("Waiting for export: no pairwise top-answer agreement table is available.")
else:
    for fig, name, caption in plots.pairwise_figures(pairs):
        save(fig, name, caption=caption)
        display_local(fig)
        plt.close(fig)
    summary = report.pairwise_table(pairs)
    summary[["model_a", "model_b"]] = summary[["model_a", "model_b"]].apply(lambda c: c.map(lambda p: style.DECISION_SHORT.get(p, p)))
    display_table("Pairwise agreement and distance per task and request mode (equal dataset weight)", summary.round(3), preview_rows=8)
    record(summary, "Pairwise agreement per task and mode, equal dataset weight")

Waiting for export: no pairwise top-answer agreement table is available.


## 4. Agreement with the consensus

How close is each model to the others? For every request, the consensus is the equal-weight mean probability distribution of all *other* models that received it (leave-one-out; at least two). Per model, dataset and request mode: the share of inputs where the model's top answer equals the consensus top answer (a tie on either side abstains) and the distance to the consensus: mean |p − p_consensus| (binary), total variation (multiclass), earth mover's distance over the ordered levels and the absolute difference of expected levels (regression). Coverage reports the share of the model's inputs with enough other models.

The consensus is not the truth, and model families can dominate it: three of the models are Qwen3.8-27B derivatives.

In [5]:
report_section("5. Agreement with the leave-one-out consensus")
loo = result.table("loo_consensus") if available else pd.DataFrame()
if loo.empty:
    print("Waiting for export: no leave-one-out consensus table is available.")
else:
    for mode in ("joint", "per_column"):
        for fig, name, caption in plots.consensus_figure(loo, mode):
            save(fig, name, caption=caption)
            display_local(fig)
            plt.close(fig)
    summary = report.equal_dataset_table(loo, report.CONSENSUS_METRICS)
    summary["provider"] = summary.provider.map(lambda p: style.DECISION_SHORT.get(p, p))
    display_table("Agreement with and distance to the consensus per task and mode (equal dataset weight)", summary.round(3), preview_rows=8)
    record(summary, "Leave-one-out consensus per task and mode, equal dataset weight")
    record(report.dataset_mode_table(loo, report.CONSENSUS_METRICS), "Leave-one-out consensus per model, dataset and mode")

Waiting for export: no leave-one-out consensus table is available.


## 5. Stand-alone quality and calibration

**Reference only — decision models are not tabular predictors.** Each model's stored *joint*-request probabilities are scored directly against the target: binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, log loss, Brier score, expected calibration error and reliability-diagram data; for regression the Spearman correlation between the expected level and the target. Nothing is trained or recalibrated, per-column requests are not scored, and rows without a joint request are reported as coverage. Labels are used only for this description and never select a model, feature or predictor.

In [6]:
report_section("6. Stand-alone quality and calibration (joint requests)")
quality = result.table("quality") if available else pd.DataFrame()
if quality.empty:
    print("Waiting for export: no stand-alone quality table is available.")
else:
    reliability = result.table("reliability")
    for fig, name, caption in direct_score_figures(quality, reliability):
        save(fig, name, caption=caption)
        display_local(fig)
        plt.close(fig)
    summary = zero_shot.summary_table(quality)
    display_table("Stand-alone scores: median and range over decision models (reference only)", summary.round(3), preview_rows=12)
    record(zero_shot.REFERENCE_LABEL + ". Labels are used only for scoring.", "Evidence boundary")
    record(summary, "Stand-alone scores per dataset (median and range over models)")
    record(quality.drop(columns="label"), "Every stand-alone score, coverage and pending artifact")
    record(reliability, "Reliability-diagram bins")

Waiting for export: no stand-alone quality table is available.


## Complete report

In [7]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified export was read; no inference, model training or live VSC changes were performed.")

exp01_outputs

1. The export read by this notebook


Export identity
{
  "status": "superseded_schema",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\exp01_outputs\\results\\outputs\\comparison.zip",
  "created_utc": "2026-11-18T13:24:39.664228+00:00",
  "origin": "verified local export of 11 complete VSC models, 2026-11-18",
  "sha256": "89c4986f83ba69546646a33b09ee4aa4ed8d94f9bd95f2d17050822d288cb8dd",
  "bytes": 1912238,
  "schema": "jevpfn.rq1a.report.v1",
  "receipt_sha256": null
}


2. Model profile


Model profile: size, readout, architecture and GPU-hours per 1,000 requests (missing = not recorded)
         model                       display_name  parameters_billion    readout_type   architecture_family  gpu_hours_per_1000_requests     compute_status
      Jev 1.14                           Jev 1.14                 NaN             api           undisclosed                          NaN waiting for export
  Liquid d1-3B               